# Module 5: PyTorch Lightning

> **Series:** PyTorch Deep Dive → PyTorch Lightning  
> **Prerequisites:** Modules 1, 2, 3 & 4


---
## 1 · PyTorch vs PyTorch Lightning

### The core problem

A raw PyTorch training loop always has the same boilerplate:

```python
for epoch in range(EPOCHS):
    model.train()
    for batch in train_dl:
        optimizer.zero_grad()
        loss = model(batch)
        scaler.scale(loss).backward()      # ← AMP boilerplate
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_()   # ← clip boilerplate
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
    model.eval()                           # ← don't forget!
    with torch.no_grad():                  # ← don't forget!
        ...
```

This is repetitive and error-prone (forgetting `model.eval()`, forgetting `torch.no_grad()`, device management across GPUs, etc.).

**PyTorch Lightning** moves all of this infrastructure into a `Trainer` object so you only write the parts that are unique to your experiment.

### PyTorch vs Lightning — decision table

| Situation | Use |
|-----------|-----|
| Learning — understanding exactly what happens each step | **Raw PyTorch** |
| Custom training logic that doesn't fit standard patterns | **Raw PyTorch** |
| Rapid iteration on model architecture / loss | **Lightning** |
| Multi-GPU / multi-node training | **Lightning** |
| Reproducible experiments with logging & checkpointing | **Lightning** |
| Research paper code that others need to rerun | **Lightning** |

Lightning is **not a different framework** — it is pure PyTorch underneath. Every `nn.Module`, every tensor operation, every `autograd` mechanism from Modules 1–3 applies unchanged. Lightning only automates the *training loop scaffolding*.

### Conceptual mapping

| Raw PyTorch | Lightning equivalent |
|-------------|---------------------|
| `model.train()` / `model.eval()` | Done automatically by `Trainer` |
| `optimizer.zero_grad()` | Done automatically |
| `loss.backward()` + `scaler` | `Trainer(precision="16-mixed")` |
| `clip_grad_norm_` | `Trainer(gradient_clip_val=1.0)` |
| `torch.no_grad()` in val loop | Done automatically |
| Moving tensors to device | `self.log(...)` + `Trainer` handles device |
| Saving checkpoints | `ModelCheckpoint` callback |
| Early stopping | `EarlyStopping` callback |

---
## 2 · Setup

In [4]:
# Install Lightning if needed
try:
    import lightning as L
    print(f"Lightning {L.__version__} already installed")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightning"])
    import lightning as L
    print(f"Installed Lightning {L.__version__}")

Lightning 2.6.1 already installed


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger

import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel

print(f"PyTorch   : {torch.__version__}")
print(f"Lightning : {L.__version__}")

# Lightning detects the device automatically — no manual device management needed
if torch.cuda.is_available():
    accelerator, devices = "cuda", 1
elif torch.backends.mps.is_available():
    accelerator, devices = "mps", 1
else:
    accelerator, devices = "cpu", 1

print(f"Accelerator: {accelerator}")

PyTorch   : 2.11.0
Lightning : 2.6.1
Accelerator: mps


---
## 3 · Data (same as Module 3)

The `Dataset` and `DataLoader` code is **identical** to Module 3 — Lightning does not change your data pipeline at all.

In [ ]:
# ── Load MS MARCO (or fall back to synthetic) ──────────────────────────────
try:
    from datasets import load_dataset
    # Download and cache locally — more reliable than streaming over a live HTTP connection.
    # The first run will take ~30 s; subsequent runs load from the HuggingFace cache instantly.
    ds = load_dataset("microsoft/ms_marco", "v1.1", split="train")

    triples = []
    for ex in ds:
        passages    = ex["passages"]
        is_selected = passages["is_selected"]
        texts       = passages["passage_text"]
        pos = [i for i, s in enumerate(is_selected) if s == 1]
        neg = [i for i, s in enumerate(is_selected) if s == 0]
        if pos and neg:
            triples.append({"query": ex["query"], "positive": texts[pos[0]], "negative": texts[neg[0]]})
        if len(triples) >= 5000:
            break
    print(f"Loaded {len(triples):,} triples from MS MARCO")
except Exception as e:
    print(f"MS MARCO unavailable ({e}) — using synthetic data")
    import random, string
    def _fake(n=60): return " ".join("".join(random.choices(string.ascii_lowercase, k=5)) for _ in range(n // 5))
    triples = [{"query": _fake(20), "positive": _fake(60), "negative": _fake(60)} for _ in range(500)]
    print(f"Using {len(triples)} synthetic triples")

# ── Tokeniser ──────────────────────────────────────────────────────────────
MODEL_NAME = "bert-base-uncased"
MAX_Q_LEN, MAX_P_LEN = 64, 128
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Dataset ────────────────────────────────────────────────────────────────
class RetrievalDataset(Dataset):
    def __init__(self, triples, tokenizer, max_q, max_p):
        self.triples, self.tokenizer = triples, tokenizer
        self.max_q, self.max_p = max_q, max_p

    def __len__(self): return len(self.triples)

    def _enc(self, text, max_len):
        return self.tokenizer(text, max_length=max_len, padding="max_length",
                              truncation=True, return_tensors="pt")

    def __getitem__(self, idx):
        item = self.triples[idx]
        q, pos, neg = self._enc(item["query"], self.max_q), self._enc(item["positive"], self.max_p), self._enc(item["negative"], self.max_p)
        return {
            "q_ids": q["input_ids"].squeeze(0),     "q_mask": q["attention_mask"].squeeze(0),
            "pos_ids": pos["input_ids"].squeeze(0), "pos_mask": pos["attention_mask"].squeeze(0),
            "neg_ids": neg["input_ids"].squeeze(0), "neg_mask": neg["attention_mask"].squeeze(0),
        }

split    = int(0.9 * len(triples))
train_ds = RetrievalDataset(triples[:split], tokenizer, MAX_Q_LEN, MAX_P_LEN)
val_ds   = RetrievalDataset(triples[split:], tokenizer, MAX_Q_LEN, MAX_P_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ms_marco' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loaded 5,000 triples from MS MARCO


KeyboardInterrupt: 

---
## 4 · The `LightningModule` — heart of Lightning

A `LightningModule` is just an `nn.Module` with a few extra methods that Lightning calls at the right time:

| Method | When called | What to put here |
|--------|------------|------------------|
| `__init__` | Once at construction | Define layers, loss, hyperparams |
| `forward` | When you call `model(x)` | Standard `nn.Module` forward pass |
| `training_step(batch, idx)` | Each training batch | Compute & return loss |
| `validation_step(batch, idx)` | Each val batch | Compute & log val metrics |
| `configure_optimizers()` | Before training starts | Return optimizer + scheduler |

That's it — Lightning calls these methods inside its training loop. You **never write** `optimizer.zero_grad()`, `loss.backward()`, or `model.train()` yourself.

### Module 3 → Module 5 diff

Here is what disappears from your code when moving to Lightning:

```python
# ❌ Gone — Lightning handles all of this
scaler = GradScaler()
model.train() / model.eval()
optimizer.zero_grad()
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer) / scaler.update()
with torch.no_grad(): ...
b = {k: v.to(device) ...}   # no manual .to(device) in training_step
```

And what you gain:

```python
# ✅ One-line Trainer config replaces all of the above
trainer = Trainer(
    precision="16-mixed",         # AMP
    gradient_clip_val=1.0,        # grad clipping
    accumulate_grad_batches=2,    # grad accumulation
    callbacks=[checkpoint, early_stop],
    logger=csv_logger,
)
```

In [ ]:
class BiEncoderLightning(L.LightningModule):
    """
    Bi-encoder from Module 3 rewritten as a LightningModule.

    Compare to Module 3 (raw PyTorch) → Module 5 (Lightning):
    - BiEncoder (nn.Module)  →  BiEncoderLightning (LightningModule)
    - Separate training loop →  training_step / validation_step
    - Separate optimizer setup → configure_optimizers
    """

    def __init__(
        self,
        model_name: str   = "bert-base-uncased",
        embedding_dim: int = 256,
        temperature: float = 0.07,
        lr: float          = 2e-5,
        weight_decay: float = 0.01,
        warmup_ratio: float = 0.1,
        total_steps: int    = 1000,
    ):
        super().__init__()
        self.save_hyperparameters()  # saves all __init__ args → self.hparams

        # ── Model ────────────────────────────────────────────────────────
        self.bert = AutoModel.from_pretrained(model_name)
        hidden    = self.bert.config.hidden_size
        self.proj = nn.Sequential(
            nn.Linear(hidden, embedding_dim),
            nn.LayerNorm(embedding_dim),
        )
        self.tau = temperature

    # ── Encoding helpers (identical to Module 3) ─────────────────────────
    @staticmethod
    def _mean_pool(token_emb: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        m = mask.unsqueeze(-1).float()
        return (token_emb * m).sum(1) / m.sum(1).clamp(min=1e-9)

    def encode(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._mean_pool(out.last_hidden_state, attention_mask)
        return F.normalize(self.proj(pooled), p=2, dim=-1)

    # ── Loss ─────────────────────────────────────────────────────────────
    def _info_nce(self, q, pos, neg):
        N       = q.size(0)
        docs    = torch.cat([pos, neg], dim=0)   # (2N, D)
        logits  = (q @ docs.T) / self.tau        # (N, 2N)
        targets = torch.arange(N, device=self.device)
        return F.cross_entropy(logits, targets)

    # ── Lightning hooks ──────────────────────────────────────────────────
    def training_step(self, batch, batch_idx):
        # NOTE: no .to(device) — Lightning moves batch to the right device automatically
        q_emb   = self.encode(batch["q_ids"],   batch["q_mask"])
        pos_emb = self.encode(batch["pos_ids"], batch["pos_mask"])
        neg_emb = self.encode(batch["neg_ids"], batch["neg_mask"])
        loss    = self._info_nce(q_emb, pos_emb, neg_emb)

        # self.log → written to logger, shown in progress bar
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        # NOTE: no torch.no_grad() — Lightning handles this automatically in val
        q_emb   = self.encode(batch["q_ids"],   batch["q_mask"])
        pos_emb = self.encode(batch["pos_ids"], batch["pos_mask"])
        neg_emb = self.encode(batch["neg_ids"], batch["neg_mask"])
        loss    = self._info_nce(q_emb, pos_emb, neg_emb)
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        warmup_steps = int(self.hparams.total_steps * self.hparams.warmup_ratio)
        decay_steps  = self.hparams.total_steps - warmup_steps
        warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps)
        decay  = LinearLR(optimizer, start_factor=1.0, end_factor=0.0,  total_iters=decay_steps)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, decay], milestones=[warmup_steps])
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }


EPOCHS     = 3
GRAD_ACCUM = 2
total_steps = len(train_dl) * EPOCHS // GRAD_ACCUM

model = BiEncoderLightning(
    model_name    = MODEL_NAME,
    embedding_dim = 256,
    temperature   = 0.07,
    lr            = 2e-5,
    total_steps   = total_steps,
)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 5 · The `Trainer` — one object replaces the training loop

The `Trainer` takes all the cross-cutting concerns that were scattered across the Module 3 loop and centralises them:

| `Trainer` argument | Replaces in Module 3 |
|--------------------|-----------------------|
| `max_epochs=3` | `for epoch in range(EPOCHS)` |
| `precision="16-mixed"` | `GradScaler` + `autocast` |
| `gradient_clip_val=1.0` | `clip_grad_norm_(model.parameters(), 1.0)` |
| `accumulate_grad_batches=2` | manual `if (i+1) % GRAD_ACCUM == 0` |
| `callbacks=[ModelCheckpoint(...)]` | manual `torch.save(...)` |
| `callbacks=[EarlyStopping(...)]` | manual `if val_loss > best: break` |
| `logger=CSVLogger(...)` | manual `train_losses.append(...)` |
| `accelerator="mps"` | `device = torch.device("mps")` + all `.to(device)` calls |

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",       # watch this metric
    mode="min",               # lower is better
    save_top_k=1,             # keep only the best checkpoint
    filename="best-{epoch:02d}-{val_loss:.4f}",
    verbose=True,
)

early_stop_cb = EarlyStopping(
    monitor="val_loss",
    patience=2,               # stop if val_loss doesn't improve for 2 epochs
    mode="min",
    verbose=True,
)

# ── Logger ───────────────────────────────────────────────────────────────────
# CSVLogger saves metrics to a CSV — no extra services needed
logger = CSVLogger("logs", name="bi_encoder")

# ── Trainer ──────────────────────────────────────────────────────────────────
# Determine precision — AMP only works on CUDA, not MPS/CPU
precision = "16-mixed" if accelerator == "cuda" else "32"

trainer = L.Trainer(
    max_epochs             = EPOCHS,
    accelerator            = accelerator,
    devices                = devices,
    precision              = precision,
    gradient_clip_val      = 1.0,
    accumulate_grad_batches= GRAD_ACCUM,
    callbacks              = [checkpoint_cb, early_stop_cb],
    logger                 = logger,
    log_every_n_steps      = 10,
    enable_progress_bar    = True,
)

print("Trainer configured. Ready to train.")
print(f"  Precision  : {precision}")
print(f"  Grad accum : {GRAD_ACCUM}  (effective batch = {BATCH_SIZE * GRAD_ACCUM})")
print(f"  Clip val   : 1.0")

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
# This single call replaces the entire ~40-line training loop from Module 3
trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)

---
## 6 · Inspecting the logs

`CSVLogger` writes metrics to `logs/bi_encoder/version_0/metrics.csv`. We can load and plot them just like in Module 3 — but without having to manually track the lists.

In [ ]:
import pandas as pd
import glob

csv_files = sorted(glob.glob("logs/bi_encoder/**/metrics.csv", recursive=True))
if csv_files:
    metrics_df = pd.read_csv(csv_files[-1])
    print(metrics_df.head())

    # Epoch-level train and val loss
    train_epoch = metrics_df.dropna(subset=["train_loss_epoch"])[["epoch", "train_loss_epoch"]]
    val_epoch   = metrics_df.dropna(subset=["val_loss"])[["epoch", "val_loss"]]

    plt.figure(figsize=(8, 3))
    plt.plot(train_epoch["epoch"], train_epoch["train_loss_epoch"], marker="o", label="Train")
    plt.plot(val_epoch["epoch"],   val_epoch["val_loss"],           marker="s", label="Val", linestyle="--")
    plt.xlabel("Epoch"); plt.ylabel("InfoNCE Loss")
    plt.title("Fine-tuning Loss Curve (Lightning)"); plt.legend(); plt.tight_layout(); plt.show()
else:
    print("No metrics CSV found yet — run the training cell first.")

---
## 7 · Loading the best checkpoint

`ModelCheckpoint` saved the best checkpoint automatically. Loading it is one line.

In [ ]:
best_ckpt = checkpoint_cb.best_model_path
print(f"Best checkpoint : {best_ckpt}")

if best_ckpt:
    best_model = BiEncoderLightning.load_from_checkpoint(best_ckpt)
    best_model.eval()
    print("Loaded best model from checkpoint.")
    print(f"Hyperparameters : {dict(best_model.hparams)}")
else:
    best_model = model
    print("Using current model (no checkpoint saved yet).")

---
## Summary

### What changed vs Module 3

| Module 3 (raw PyTorch) | Module 5 (Lightning) |
|------------------------|----------------------|
| `BiEncoder(nn.Module)` + separate training loop (~40 lines) | `BiEncoderLightning(LightningModule)` with `training_step` / `validation_step` |
| Manual `GradScaler`, `autocast` | `Trainer(precision="16-mixed")` |
| Manual `clip_grad_norm_` | `Trainer(gradient_clip_val=1.0)` |
| Manual gradient accumulation counter | `Trainer(accumulate_grad_batches=2)` |
| `model.train()` / `model.eval()` | Automatic |
| `torch.no_grad()` in val | Automatic |
| Manual `train_losses.append(...)` | `self.log(...)` → CSV |
| Manual `torch.save(...)` | `ModelCheckpoint` callback |
| No early stopping | `EarlyStopping` callback |

The model weights, loss function, and data pipeline are **byte-for-byte identical** — only the loop scaffolding changed.

### When to pick each

- **Learning / debugging** → raw PyTorch (Modules 1–4): full visibility into every step.
- **Running experiments** → Lightning (Module 5): less boilerplate, reproducible, scalable to multi-GPU with zero code changes.